# Drive, github and requirements settings

In [1]:
from google.colab import drive
from google.colab import userdata


drive.mount('/content/drive')
KEY = userdata.get('KEY')

Mounted at /content/drive


set Github access token

In [ ]:
import os
from google.colab import userdata
github_access_token = userdata.get('GITHUB_TOKEN')

# Replace 'your_token_here' with your actual token and 'your_repo_url_here' with your repository URL
os.environ['GITHUB_TOKEN'] = github_access_token

repo_url = 'https://github.com/sustaz/principle_of_law_detection.git'
modified_url = repo_url.replace('https://', f'https://{os.environ["GITHUB_TOKEN"]}@')

Clone Repository

In [ ]:
!git clone {modified_url}

Cloning into 'principle_of_law_detection'...
remote: Enumerating objects: 60, done.
remote: Counting objects: 100% (60/60), done.
remote: Compressing objects: 100% (47/47), done.
remote: Total 60 (delta 29), reused 40 (delta 12), pack-reused 0 (from 0)
Receiving objects: 100% (60/60), 152.58 KiB | 2.46 MiB/s, done.
Resolving deltas: 100% (29/29), done.


Set github credentials

In [ ]:
!chmod +x ./principle_of_law_detection/bash_commands/set_git_credentials.sh
!./principle_of_law_detection/bash_commands/set_git_credentials.sh

Syncronize notebook version

In [ ]:
!cp /content/drive/MyDrive/POLINE/gpt_notebook.ipynb /content/principle_of_law_detection/

In [ ]:
!cp /content/drive/MyDrive/POLINE/old_test_outputs.ipynb /content/principle_of_law_detection/

Add, commit, push code

In [ ]:
!chmod +x ./principle_of_law_detection/bash_commands/add_commit_push.sh
!./principle_of_law_detection/bash_commands/add_commit_push.sh "cleaned gpt notebook and updated library"

[main 4a06a77] cleaned gpt notebook and updated library
 1 file changed, 2 insertions(+), 1 deletion(-)
Enumerating objects: 5, done.
Counting objects: 100% (5/5), done.
Delta compression using up to 2 threads
Compressing objects: 100% (3/3), done.
Writing objects: 100% (3/3), 308 bytes | 308.00 KiB/s, done.
Total 3 (delta 2), reused 0 (delta 0), pack-reused 0
remote: Resolving deltas: 100% (2/2), completed with 2 local objects.
To https://github.com/sustaz/principle_of_law_detection.git
   4c54f2c..4a06a77  main -> main


Install requirements

In [ ]:
!pip install -r '/content/principle_of_law_detection/requirements.txt'

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 244.3/244.3 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 386.9/386.9 kB 9.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 20.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 255.8/255.8 kB 12.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.5/27.5 MB 59.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.6/232.6 kB 12.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.4/76.4 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.0/78.0 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 325.2/325.2 kB 19.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.3/58.3 kB 3.3 MB/s eta 0:00:00


# Import libraries

In [ ]:
from principle_of_law_detection.src import gpt_utils as gu, utils as sr, text_preprocessing as tp, evaluation as ev
import json
import os
import pandas as pd

# Single experiments

## Single prompt approach

In [ ]:
def prompt_single_trial1(txt):
  return f"""
  Extract portions of the text from the argumentative part of the judgment that fit the definition of a JPOL.

    {txt}

    A JPOL (Judicial Principle of Law) should:

    Source and Content:
        Be a portion of text extracted from the argumentative part of a judgment.
        Contain the interpretation provided by the deciding court or an endorsed interpretation from a previous decision.

    Nature of Interpretation:
        Interpretate a rule, a general principle, or the consequences stemming from the application of a rule or principle within the legal system.
        Not be a rephrase of the legislation or a previous paragraph.
        Not concern the application of facts to the current case.
        Not be what the referring court asks.

    Citations and Endorsements:
        A JPOL can cite another JPOL.
        A JPOL can endorse a precedent of a European Court or an Advocate General's statement.

    Classify each paragraph with Y if the paragraph is a JPOL, N if the paragraph is not a JPOL.
    Avoid any explanation into the output.
    Use the following format for the output:
      Paragraph number: class

      Here are a few examples of what is NOT a JPOL:
  37
  On the contrary, the consequence of such a condition is to exclude altogether any reduction of the taxable amount for VAT purposes in the case of unpaid claims arising during the six-month period preceding the declaration of insolvency of the debtor company concerned, even where those claims become definitively irrecoverable at the end of the insolvency proceedings. Such automatic refusal of the right to a reduction is contrary to the principle of the neutrality of VAT.
  30
  As regards the context of which Article 132(1)(b) of the VAT Directive forms part, it is important to note that that provision must be read in the light of Article 134(a) of that directive, which requires, in any event, that the supply of goods or services concerned be essential to the transactions exempted within the scope of hospital and medical care (see, to that effect, judgments of 1 December 2005, Ygeia, C-394/04 and C-395/04, EU:C:2005:734, paragraph 26; of 14 June 2007, Horizon College, C-434/05, EU:C:2007:343, paragraph 38; and of 8 October 2020, Finanzamt D, C-657/19, EU:C:2020:811, paragraph 31).
  54
  In the light of the discretion enjoyed by the Member States in that context, as noted in paragraph 40 above, the Court has held that the existence of the option provided for in the first paragraph of Article 133 of the VAT Directive supports the interpretation that it is for the national law of each Member State to lay down the rules according to which such recognition may be granted to establishments which request it, even if the fact that a Member State has not exercised that option does not affect the possibility that an establishment may be recognised for the purposes of granting the exemption referred to in Article 132(1)(b) of the VAT Directive (see, to that effect, judgment of 6 November 2003, Dornier, C-45/01, EU:C:2003:595, paragraphs 64 to 66).

"""

system_prompt = "You are a judge with strong knowledge on tax law, expert about extracting Judicial Principles of Law (JPOLs) from legal judgments. You are very very skeptical and tend to say something is NOT a JPOL"

In [ ]:
def prompt_single_trial2(txt):
  return f"""
    1. A JPOL is a portion of text, extracted from the argumentative part of a judgement which contains the interpretation provided by the deciding court or provided in
     a previous decision and endorsed by the deciding court;
  This interpretation concerns a rule; or a general principle; or focuses on the consequences stemming from the application of a rule or a principle in a legal system.
  2. A JPOL can be a citation of another JPOL taken from a previous judgement AND
  3. A JPOL is not a rephrase of the legislation or a previous paragraph AND
  4. A JPOL is not a question concerning the application of facts to the current case AND
  5. A JPOL can be the endorsement of a precedent of a European Court AND
  6. A JPOL can be the endorsement of an Advocate General's statement AND
  7. A JPOL is not what the referring court asks.

  For each JPOL all the conditions must apply.

  Consider the following judgement, where each paragraph starts with a number:

  {txt}

  Check if each paragraph has the characteristics of a JPOL.

  Use the following format:
  Paragraph number: Y if JPOL.
  Paragraph number: N if not JPOL.
  Paragraph number: UND if it meets both criteria.

  Here are a few examples of what is NOT a JPOL:

  On the contrary, the consequence of such a condition is to exclude altogether any reduction of the taxable amount for VAT purposes in the case of unpaid claims arising during the six-month period preceding the declaration of insolvency of the debtor company concerned, even where those claims become definitively irrecoverable at the end of the insolvency proceedings. Such automatic refusal of the right to a reduction is contrary to the principle of the neutrality of VAT.

  As regards the context of which Article 132(1)(b) of the VAT Directive forms part, it is important to note that that provision must be read in the light of Article 134(a) of that directive, which requires, in any event, that the supply of goods or services concerned be essential to the transactions exempted within the scope of hospital and medical care (see, to that effect, judgments of 1 December 2005, Ygeia, C-394/04 and C-395/04, EU:C:2005:734, paragraph 26; of 14 June 2007, Horizon College, C-434/05, EU:C:2007:343, paragraph 38; and of 8 October 2020, Finanzamt D, C-657/19, EU:C:2020:811, paragraph 31).

  In the light of the discretion enjoyed by the Member States in that context, as noted in paragraph 40 above, the Court has held that the existence of the option provided for in the first paragraph of Article 133 of the VAT Directive supports the interpretation that it is for the national law of each Member State to lay down the rules according to which such recognition may be granted to establishments which request it, even if the fact that a Member State has not exercised that option does not affect the possibility that an establishment may be recognised for the purposes of granting the exemption referred to in Article 132(1)(b) of the VAT Directive (see, to that effect, judgment of 6 November 2003, Dornier, C-45/01, EU:C:2003:595, paragraphs 64 to 66).

  To recognise such an insurer as having that status would be tantamount to disregarding the principle of fiscal neutrality, since the VAT paid to the tax authorities would not be exactly proportional to the price actually received by the taxable customers who carried out the taxable transactions in question.
"""

system_prompt = ""

In [ ]:
def jpol_prompt_no_par_4_single(txt):
  prompt =  f"""

    Extract portions of the text from the argumentative part of the judgment that fit the definition of a JPOL.
    Only certain portions of the judgement text contain JPOLS, and these portions must be at least one complete sentence.
    A JPOL (Judicial Principle of Law) should fit all the following conditions:

    1. A JPOL is a portion of text, extracted from the argumentative part of a judgement which contains the interpretation provided by the deciding court or provided in
     a previous decision and endorsed by the deciding court; This interpretation concerns a rule; or a general principle; or focuses on the consequences stemming from the application of a rule or a principle in a legal system
    2. A JPOL can be a citation of another JPOL taken from a previous judgement AND
    3. A JPOL is not a rephrase of the legislation or a previous paragraph AND
    4. A JPOL is not a question concerning the application of facts to the current case AND
    5. A JPOL can be the endorsement of a precedent of a European Court AND
    6. A JPOL can be the endorsement of an Advocate General's statement AND
    7. A JPOL is not what the referring court asks.

    For each JPOL all the conditions must apply.

    #####

    Argumentative part of the judgment:

    {txt}

    #####
    Please extract a list of JPOLs using the format below.

    - A JPOL is a complete sentence found in the judgement.
    - It starts with a capital letter and ends with an end punctuation mark (.)

    #####

    Instructions for output:
    - Tag the portions of text that fit the JPOL definition with <JPOL> and </JPOL>.
    - Inside the <JPOL> tag, return only the first five and the last five words of the sentence.
    - Separate the first five and the last five words with ellipses (.....).

    Example output:
    If you have a sentence like this:
    This is just an example on how to tag a JPOL in a sentence.

    The tagged JPOL should be:
    <JPOL>This is just an example.....a JPOL in a sentence.</JPOL>

    """

  return prompt


system_prompt_no_par = "You are an expert about Judicial Principles of Law (JPOLs) from legal judgments."

In [ ]:
response = gu.ask_gpt_2(jpol_prompt_no_par_4_single(txt), system_prompt, KEY, model="gpt-4o", max_tokens=4000, temperature=0.2, top_p=1)

In [ ]:
txt = tp.extract_text_between_markers(txt)

In [ ]:
file = "/content/drive/MyDrive/POLINE/Dataset/Dataset_V1/Preprocessed_Judgements_Subset_noparagraph/Copia di VDP Dental Laboratory NV v Staatssecretaris van FinanciÃ«n a.xml"

with open(file) as f:
    file = f.read()

txt = "".join(file)

response = gu.ask_gpt_2(jpol_prompt_no_par_4_single(txt), system_prompt, KEY, model="gpt-4o", max_tokens=4000, temperature=0.2, top_p=1)

## LLM + Sim

In [ ]:
import pandas as pd

# Replace 'your_excel_file.xlsx' with the actual file path
jpol_labels = pd.read_excel('/content/drive/MyDrive/POLINE/JPOL_labels.xlsx')['LABELS'].to_list()

In [ ]:
def prompt_rag(txt):
  return f"""
          You are a professionist judge with strong knowledge on tax law.
          Your task is to identify JPOLS inside a text of a judgment.

          DEFINITION OF A JPOL:
              All JPOLs should Contain the interpretation provided by the deciding court or an endorsed interpretation from a previous decision.

              Nature of Interpretation:
                  - Interpretate a rule, a general principle, or the consequences stemming from the application of a rule or principle within the legal system.
                  - Not concern the application of facts to the current case.
                  - Not be what the referring court asks.

              Citations and Endorsements:
                  - A JPOL can cite another JPOL.
                  - A JPOL can endorse a precedent of a European Court or an Advocate General's statement.

          OUTPUT:
            The scope is to have as output a list of attributes that are useful, in a second phase, to find all the chunks in the text judgements
            that are JPOL, trough BM25 or cosine similarity. In the output include:
              - JPOL label
              - Motivation according to the definition
              - keywords

          TEXT OF THE JUDGEMENT:

          {txt}

            """


def prompt_rag_2(txt, jpol_labels):
  return f"""

          The scope is to have as output a list of attributes that are useful, in a second phase, to find all the chunks in the text judgements
          that are JPOL, trough BM25 or cosine similarity. In the output include:
            - JPOL label
            - Motivation according to the definition
            - keywords

            To assign the labels, use the following list of JPOLS LABELS:

            {jpol_labels}


          TEXT OF THE JUDGEMENT:

          {txt}

            """



system_prompt = """You are a professionist judge with strong knowledge on tax law.
                    Your task is to identify JPOLS inside a text of a judgment.

                    DEFINITION OF A JPOL:
                        All JPOLs should Contain the interpretation provided by the deciding court or an endorsed interpretation from a previous decision.

                        Nature of Interpretation:
                            - Interpretate a rule, a general principle, or the consequences stemming from the application of a rule or principle within the legal system.
                            - Not concern the application of facts to the current case.
                            - Not be what the referring court asks.

                        Citations and Endorsements:
                            - A JPOL can cite another JPOL.
                            - A JPOL can endorse a precedent of a European Court or an Advocate General's statement."""

In [ ]:
file_path = "/content/drive/MyDrive/POLINE/Dataset/Dataset_V1/Preprocessed_Judgements_Subset/Almos AgrÃ¡rkÃ¼lkereskedelmi Kft v Nemzeti AdÃ³- Ã©s VÃ¡mhiv.xml"

with open(file_path) as f:
    file = f.read()

txt = "".join(file)

response = gu.ask_gpt_2(prompt_rag_2(txt, jpol_labels), system_prompt, KEY, model="gpt-4o", max_tokens=4096, temperature=0.2, top_p=1)

In [ ]:
print(response)

### JPOL Identification and Analysis

#### JPOL 1
- **JPOL Label**: Concept of ‘subsidy directly linked to the price’
- **Motivation**: This interpretation clarifies the conditions under which a directive can be considered to have been correctly transposed into national law, emphasizing that the exact wording is not necessary as long as the directive's objectives are met in a clear and precise manner.
- **Keywords**: transposition, directive, national law, clear and precise manner, full application

#### JPOL 2
- **JPOL Label**: Concept of "non payment of the price"
- **Motivation**: This interpretation discusses the conditions under which Member States must reduce the taxable amount for VAT due to non-payment, cancellation, or reduction in price, highlighting the fundamental principle that VAT should only be charged on the amount actually received.
- **Keywords**: Article 90(1), VAT Directive, non-payment, cancellation, reduction in price, taxable amount

#### JPOL 3
- **JPOL Label**:

In [ ]:
import re


def extract_jpol_data(text):
    # Pattern to capture each JPOL block
    jpol_pattern = re.compile(r'### JPOL \d+\s*\n- \*\*JPOL Label\*\*: (.*?)\s*\n- \*\*Motivation\*\*: (.*?)\s*\n- \*\*Keywords\*\*: (.*?)(?=\n###|$)', re.DOTALL)

    # Find all matches
    matches = jpol_pattern.findall(text)

    # Create a list of dictionaries for each JPOL (Label, Motivation, Keywords)
    jpol_list = []
    for label, motivation, keywords in matches:
        jpol_dict = {
            "Label": label.strip(),
            "Motivation": motivation.strip(),
            "Keywords": [kw.strip() for kw in keywords.split(',')]
        }
        jpol_list.append(jpol_dict)

    return jpol_list

In [ ]:
extracted_jpols = extract_jpol_data(response)

In [ ]:
extracted_jpols

[{'Label': 'Concept of ‘subsidy directly linked to the price’',
  'Motivation': "This interpretation clarifies the conditions under which a directive can be considered to have been correctly transposed into national law, emphasizing that the exact wording is not necessary as long as the directive's objectives are met in a clear and precise manner.",
  'Keywords': ['transposition',
   'directive',
   'national law',
   'clear and precise manner',
   'full application']},
 {'Label': 'Concept of "non payment of the price"',
  'Motivation': 'This interpretation discusses the conditions under which Member States must reduce the taxable amount for VAT due to non-payment, cancellation, or reduction in price, highlighting the fundamental principle that VAT should only be charged on the amount actually received.',
  'Keywords': ['Article 90(1)',
   'VAT Directive',
   'non-payment',
   'cancellation',
   'reduction in price',
   'taxable amount']},
 {'Label': 'Debt which has become definitively

In [ ]:
import os
import xml.etree.ElementTree as ET
from sentence_transformers import SentenceTransformer
import numpy as np
import faiss
from bs4 import BeautifulSoup
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import linear_kernel
import re
import PyPDF2

def extract_text_from_pdf(pdf_path):
    # Open the PDF file
    with open(pdf_path, 'rb') as file:
        reader = PyPDF2.PdfReader(file)
        text = ""
        # Extract text from all pages
        for page_num in range(len(reader.pages)):
            text += reader.pages[page_num].extract_text()
    return text

def find_article(text, article_number, subsection=None):
    # Regular expression to match articles more robustly
    if subsection:
        pattern = rf'Article\s*{article_number}\s*\({subsection}\)[\s\S]+?(?=Article\s*\d+\s*\(?[a-z]?\)?\s|$)'
    else:
        pattern = rf'Article\s*{article_number}\s*[\s\S]+?(?=Article\s*\d+\s*\(?[a-z]?\)?\s|$)'

    match = re.search(pattern, text, re.IGNORECASE)
    if match:
        return match.group().strip()
    else:
        return f"Article {article_number} {'(' + subsection + ')' if subsection else ''} not found."



# Step 1: Parse XML and Extract Paragraphs
def parse_xml_to_paragraphs(file_path):
    with open(file_path, 'r') as file:
        # Using BeautifulSoup for parsing the XML file
        soup = BeautifulSoup(file, 'lxml')
        text_content = soup.get_text(separator=" ").strip()

        # Splitting the text into paragraphs based on the pattern (paragraph number)
        paragraphs = []
        current_paragraph = ""

        for line in text_content.splitlines():
            line = line.strip()
            if line.isdigit():  # It's a paragraph number
                if current_paragraph:
                    paragraphs.append(current_paragraph.strip())
                current_paragraph = ""
            else:
                current_paragraph += " " + line

        if current_paragraph:
            paragraphs.append(current_paragraph.strip())

    return paragraphs

# Step 2: Generate embeddings for paragraphs
def create_embeddings(paragraphs, model):
    return model.encode(paragraphs)

# Step 3: Create FAISS Index for Efficient Search
def create_faiss_index(embeddings):
    dimension = embeddings.shape[1]
    index = faiss.IndexFlatL2(dimension)
    index.add(embeddings)
    return index

# Step 4: Search the most similar paragraph using the query
def search_similar_paragraph(query_embedding, index, paragraphs, k):
    D, I = index.search(query_embedding, k)  # Distance and index
    return [(paragraphs[idx], D[0][i]) for i, idx in enumerate(I[0])]

# Step 5: Create Query Embedding based on Label, Motivation, and Keywords
def create_query_embedding(label, motivation, keywords, model):
    query_text = f"{label}"
    return model.encode([query_text])


def cosine_similarity_search(query, tfidf_matrix, paragraphs, k):
    # Transform the query to match the same TF-IDF dimensions
    query_tfidf = vectorizer.transform([query])

    # Compute cosine similarity between the query TF-IDF vector and all paragraph TF-IDF vectors
    cosine_similarities = linear_kernel(query_tfidf, tfidf_matrix).flatten()


    # Get the indices of the paragraphs with the highest cosine similarities
    top_k_indices = cosine_similarities.argsort()[-k:][::-1]  # argsort sorts in ascending; reverse for descending

    # Return the top k similar paragraphs with their indices and similarities
    return [(paragraphs[idx], cosine_similarities[idx]) for idx in top_k_indices]


def get_max_score_texts(text_tuples):
    # Dictionary to hold unique texts with all their scores
    score_dict = {}

    # Iterate through each tuple in the list
    for text, score in text_tuples:
        if text in score_dict:
            # Append the score to the list of scores for this text
            score_dict[text].append(score)
        else:
            # Start a new list of scores for this text
            score_dict[text] = [score]

    # List to hold texts with their highest scores
    max_score_texts = []

    # Find the text with the maximum score among its duplicates
    for text, scores in score_dict.items():
        max_score = max(scores)  # Find the maximum score for the text
        max_score_texts.append((text, max_score))  # Append tuple of text and its max score

    return max_score_texts


def flatten_list(list_of_lists):
    # Using list comprehension to flatten the list
    return [element for sublist in list_of_lists for element in sublist]

In [ ]:
# Example usage
pdf_path = '/content/drive/MyDrive/POLINE/CELEX_32006L0112_EN_TXT.pdf'  # Adjust the path to your file
text = extract_text_from_pdf(pdf_path)
article_number = 138  # Replace with the desired article number
subsection = None  # Replace with the desired subsection, or None if you want the entire article
result = find_article(text, article_number, subsection)

print(result)

Article 138, goods dispatched or transported to a Member Stateother than that in which dispatch or transport of the goodsbegins are supplied VAT-exempt or where goods are transferredVAT-exempt to another Member State by a taxable person for thepurposes of his business, VAT shall become chargeable on the15th day of the month following that in which the chargeableevent occurs.
2. By way of derogation from paragraph 1, VAT shall become
chargeable on issue of the invoice provided for in Article 220, ifthat invoice is issued before the 15th day of the month followingthat in which the chargeable event occurs.
CHAPTER 3
Intra-Community acquisition of goods


In [ ]:
model = SentenceTransformer('all-MiniLM-L6-v2')

paragraphs = parse_xml_to_paragraphs(file_path)

# Create the TF-IDF model
vectorizer = TfidfVectorizer()
tfidf_matrix = vectorizer.fit_transform(paragraphs)

#paragraph_embeddings = create_embeddings(paragraphs, model)

#index = create_faiss_index(np.array(paragraph_embeddings))
similar_paragraphs = []

for jpol in extracted_jpols:

    label = jpol['Label']
    motivation = jpol['Motivation']
    keywords = jpol['Keywords']

    #query_embedding = create_query_embedding(label, motivation, keywords, model)

    #similar_paragraphs = search_similar_paragraph(query_embedding, index, paragraphs, k=5)

    query = f"{label}"
    similar_paragraphs.append(cosine_similarity_search(query, tfidf_matrix, paragraphs, 5))



flattened_results = flatten_list(similar_paragraphs)
results = get_max_score_texts(flattened_results)

# Output results
for text, score in results:
  print(f"Text: {text}\nScore: {score}\n")

Text: Secondly, it is important, on the other hand, that, for situations other than those linked to the non-payment of the price, national transposing provisions take into account all the situations in which, after a transaction has been concluded, part or all of the consideration has not been received by the taxable person, which is a matter for the national court to ascertain.
Score: 0.3189775567427321

Text: However, Article 90(2) permits Member States to derogate from the abovementioned rule in the case of total or partial non-payment of the transaction price. Hence taxable persons cannot rely, under Article 90(1) of the VAT Directive, on a right to a reduction of their taxable amount for VAT in the case of non-payment of the price if the Member State concerned intended to apply the derogation provided for in Article 90(2) of that directive.
Score: 0.5222436523443988

Text: It must be noted in that regard that, if the total or partial non-payment of the purchase price occurs withou

In [ ]:
flattened_results[1]

0.4263512099936439

#PREPROCESSING

In [ ]:
from principle_of_law_detection.src import text_preprocessing as pr
import re


judgements_root = "/content/drive/MyDrive/POLINE/Dataset/Dataset_V1/Judgements_Subset" # file originali

judgements_root_output = "/content/drive/MyDrive/POLINE/Dataset/Dataset_V2/Judgements_Subset_by_paragraphs"

os.makedirs(judgements_root_output, exist_ok=True)

judgements = os.listdir(judgements_root)


for judgement in judgements:

  with open(os.path.join(judgements_root, judgement)) as f:
    file = f.read()

  txt = "".join(file)

  #txt = pr.extract_text_between_markers(txt)

  cleaned_paragraps = []
  # Find all the paragraphs
  #paragraphs = re.split(r'\n\n\d+\n\n', txt)

  # Define the regex pattern
  pattern = re.compile(r'(?<=\n\n)(\d+)')

  # Split the text using the regex pattern
  paragraphs = pattern.split(txt.strip())

  paragraphs_indexed = ['\n\nPARAGRAPH NUMBER : ' + paragraphs[i] + ' TEXT: ' + paragraphs[i+1] + '\n\n' for i in range(1, len(paragraphs), 2)]

  for par in paragraphs_indexed:

    """if 'stated in paragraph' in par.lower() or 'referring court asks' in par.lower():
      continue
    else:"""
    cleaned_paragraps.append(par.strip())

    # modulino che si può potenziare eventualmetne con similarity dividendo i chunk per phrases

  txt = "\n\n".join(cleaned_paragraps)

  with open(os.path.join(judgements_root_output, judgement), 'w') as f:
    f.write(txt)

# MASSIVE EXPERIMENTS

In [ ]:
def jpol_prompt(txt):
  prompt =  f"""Extract portions of the text from the argumentative part of the judgment that fit the definition of a JPOL.

    {txt}

    A JPOL (Judicial Principle of Law) should:

    Source and Content:
        Be a portion of text extracted from the argumentative part of a judgment.
        Contain the interpretation provided by the deciding court or an endorsed interpretation from a previous decision.

    Nature of Interpretation:
        Interpretate a rule, a general principle, or the consequences stemming from the application of a rule or principle within the legal system.
        Not be a rephrase of the legislation or a previous paragraph.
        Not be a question concerning the application of facts to the current case.
        Not be what the referring court asks.

        Citations and Endorsements:
        A JPOL can cite another JPOL.
        A JPOL can endorse a precedent of a European Court or an Advocate General's statement.

    Use the following format for the output:
      Paragraph number: Y if JPOL.
      Paragraph number: N if not JPOL."""

  return prompt


system_prompt = "You are an expert of judge with strong knowledge on jurisdiction and tax law. "

In [ ]:
jsons_root = "/content/drive/MyDrive/POLINE/Annotazioni/poline_jsons/"
annotations_files = os.listdir(jsons_root)

In [ ]:
judgements_root = "/content/drive/MyDrive/POLINE/Dataset/Dataset_V2/Preprocessed_Judgements_Subset"
judgemnts = os.listdir(judgements_root)

In [ ]:
judgemnts

['A & G Fahrschul-Akademie GmbH v Finanzamt Wolfenbüttel.xml',
 'Autoridade Tributária e Aduaneira v Termas Sulfurosas de Alcafache SA.xml',
 'Boehringer Ingelheim RCV GmbH & Co. KG Magyarországi Fióktelepe v Nemzeti Adó- és Vámhivatal Fellebbviteli Igazgatósága.xml',
 'CS and Finanzamt Österreich, Dienststelle Graz-Stadt v Finanzamt Österreich, Dienststelle Judenburg Liezen and technoRent International GmbH.xml',
 'DNB Banka AS v Valsts ieņēmumu dienests.xml',
 'ELVOSPOL s.r.o. v Odvolací finanční ředitelství.xml',
 'Euler Hermes SA Magyarországi Fióktelepe v Nemzeti Adó- és Vámhivatal Fellebbviteli Igazgatósága.xml',
 'Finanzamt B v X-Beteiligungsgesellschaft mbH.xml',
 'I GmbH v Finanzamt H.xml',
 'Michael Winterhoff v Finanzamt Ulm and Jochen Eisenbeisvv Bundeszentralamt für Steuern.xml',
 'Minister Finansów v Aviva Towarzystwo Ubezpieczeń na Życie S.A. w Warszawie.xml']

In [ ]:
prompt_name = "p_aspries_2"
responses = []

for idx, judgement in enumerate(judgemnts):

  with open(os.path.join(judgements_root, judgement)) as f:
    file = f.read()

  txt = "".join(file)

  response = gu.ask_gpt_2(jpol_prompt(txt), system_prompt, KEY, model="gpt-4o", max_tokens=4000, temperature=0.2, top_p=1)

  result_root = f"/content/drive/MyDrive/POLINE/Results/preprocessed_input/Dataset_V2/full_responses_{prompt_name}"
  os.makedirs(result_root, exist_ok=True)
  sr.write_text_to_docx(response, os.path.join(result_root, f"{judgement}_full_response.docx"))

  responses.append((response, judgement))

In [ ]:
def extract_paragraphs(text):
    # Regular expression to find pairs of number and answer (Y or N)
    pattern = r'(\d+): (Y|N)'
    matches = re.findall(pattern, text)

    # Convert matches to list of tuples
    #pairs = [(int(num), answer) for num, answer in matches]

    return pd.DataFrame(matches, columns=['paragraph_number', 'label'])

Save results

In [ ]:
import re
dfs = []
for response, file_name in responses:
    dfs.append((extract_paragraphs(response.replace("<", "").replace(">", "")), file_name[:5]))

results_df = sr.concatenate_dataframes(dfs)
#results_df.to_excel(f"/content/drive/MyDrive/POLINE/Results/preprocessed_input/{prompt_name}_remaining.xlsx", index=False)

In [ ]:
results_df

,paragraph_number,label,file_name
0,17,N,A & G
1,18,Y,A & G
2,19,Y,A & G
3,20,N,A & G
4,21,N,A & G
...,...,...,...
291,36,Y,Minis
292,37,Y,Minis
293,38,Y,Minis
294,39,N,Minis


# EVALUATION STRUCTURED IN PARAGRAPHS

In [ ]:
jsons_root = "/content/drive/MyDrive/POLINE/Annotazioni/poline_jsons/"
annotations_files = os.listdir(jsons_root)

In [ ]:
judgements_root = "/content/drive/MyDrive/POLINE/Dataset/Dataset_V2/Preprocessed_Judgements_Subset"
judgemnts = os.listdir(judgements_root)

Read GT from Json

In [ ]:
ground_truths = []

for ann_file in annotations_files:

  ann_dict = json.load(open(os.path.join(jsons_root,ann_file)))


  for ann in ann_dict['annotations']:
    paragraph_number = ann['text'].split()[0]
    # split_par = str(int(paragraph_number) + 1)
    # txt_par = txt.split(split_par)
    label = ann['type']
    file_name = ann_file[:5]

    ground_truths.append((file_name, paragraph_number, label))

gt_batch1 = pd.DataFrame(ground_truths, columns=['file_name', 'paragraph_number', 'ground_truth'])

In [ ]:
gt_batch1

,file_name,paragraph_number,ground_truth
0,ELVOS,25,JPOL
1,ELVOS,27,JPOL
2,ELVOS,28,JPOL
3,ELVOS,29,JPOL
4,ELVOS,31,JPOL
...,...,...,...
140,Euler,33,JPOL
141,Euler,34,JPOL
142,Euler,35,JPOL
143,Euler,38,JPOL


Read GT from excel

In [ ]:
gt_batch2 = pd.read_excel('/content/drive/MyDrive/POLINE/Annotazioni/CJUE_Taxable_Amount_Addendum_Piera.xlsx')
gt_batch2['file_name'] = gt_batch2['file_name'].apply(lambda x: x[:5])
gt_batch2['ground_truth'] = 'JPOL'
gt_batch2

In [ ]:
ground_truth_df = pd.concat([gt_batch1, gt_batch2]).drop('other', axis=1)

Merge annotations with original files to have all NOT JPOLS paragraphs

In [ ]:
import os
import pandas as pd

# Folder containing text files
folder_path = 'drive/MyDrive/POLINE/Dataset/Dataset_V2/Judgements_Subset_by_paragraphs'

def merge_annotations_with_files(folder_path, df):
  # Function to parse text files
  def parse_paragraphs(file_path):
      paragraphs = {}
      with open(file_path, 'r') as f:
          content = f.read().split('PARAGRAPH NUMBER : ')
          for part in content[1:]:
              number, text = part.split('TEXT:', 1)
              paragraphs[int(number.strip())] = text.strip()
      return paragraphs

  # List to collect new rows
  new_rows = []

  # Iterate over each file
  for file_name in df['file_name'].unique():
      # Get the corresponding text file
      matching_files = [f for f in os.listdir(folder_path) if f.startswith(file_name[:5])]

      for txt_file in matching_files:
          txt_file_path = os.path.join(folder_path, txt_file)

          # Parse the paragraphs from the text file
          paragraphs = parse_paragraphs(txt_file_path)

          # Get the paragraph numbers already in the DataFrame for this file
          existing_paragraphs = df[df['file_name'] == file_name]['paragraph_number'].tolist()

          # Find missing paragraphs
          actual_paragraphs = {str(num) for num in paragraphs.keys()}
          missing_paragraphs = set(actual_paragraphs) - set(existing_paragraphs)

          print(missing_paragraphs)
          # Add missing paragraphs to the list of new rows
          for paragraph_number in missing_paragraphs:
              new_row = {
                  'file_name': file_name,
                  'paragraph_number': paragraph_number,
                  'ground_truth': 'not_JPOL'
              }
              new_rows.append(new_row)

  # Create a DataFrame from the new rows and concatenate it to the original DataFrame
  if new_rows:
      df_new = pd.DataFrame(new_rows)
      gt_df = pd.concat([df, df_new], ignore_index=True)

  return gt_df

# Sort the dataframe by file_name and paragraph_number
ground_truth_df = merge_annotations_with_files(folder_path, gt_batch1)

{'40', '11', '26', '24', '3', '17', '16', '44', '7', '19', '18', '12', '37', '1', '20', '34', '43', '13', '10', '36', '15', '35', '42', '30', '2', '4', '38', '6', '8', '33', '39', '46', '48', '21', '14', '22', '23', '5', '9', '45'}
{'40', '11', '24', '41', '3', '17', '16', '7', '27', '19', '29', '28', '56', '18', '12', '49', '37', '1', '20', '13', '51', '10', '62', '63', '15', '35', '42', '53', '65', '23', '30', '2', '4', '57', '25', '6', '8', '33', '60', '46', '48', '21', '14', '22', '26', '5', '9', '45', '32'}
{'24', '3', '7', '27', '29', '28', '56', '18', '12', '37', '20', '13', '51', '62', '15', '58', '64', '30', '38', '57', '25', '31', '48', '21', '14', '26', '43', '40', '11', '47', '17', '16', '19', '1', '10', '36', '35', '54', '32', '2', '4', '6', '8', '33', '50', '67', '46', '52', '22', '23', '5', '9', '34'}
{'11', '24', '3', '17', '16', '44', '7', '19', '27', '18', '12', '1', '20', '13', '10', '15', '4', '38', '2', '25', '6', '8', '31', '39', '46', '21', '14', '22', '23', '5',

In [ ]:
gt_batch1

,file_name,paragraph_number,ground_truth
0,ELVOS,25,JPOL
1,ELVOS,27,JPOL
2,ELVOS,28,JPOL
3,ELVOS,29,JPOL
4,ELVOS,31,JPOL
...,...,...,...
140,Euler,33,JPOL
141,Euler,34,JPOL
142,Euler,35,JPOL
143,Euler,38,JPOL


In [ ]:
ground_truth_df

,file_name,paragraph_number,ground_truth
0,ELVOS,25,JPOL
1,ELVOS,27,JPOL
2,ELVOS,28,JPOL
3,ELVOS,29,JPOL
4,ELVOS,31,JPOL
...,...,...,...
601,Euler,22,not_JPOL
602,Euler,23,not_JPOL
603,Euler,5,not_JPOL
604,Euler,9,not_JPOL


In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score

def compute_metrics(ground_truth_df, results_df):
    res_df = results_df.copy()
    gt_df = ground_truth_df.copy()

    gt_df['paragraph_number'] = gt_df['paragraph_number'].astype(str)
    res_df['paragraph_number'] = res_df['paragraph_number'].astype(str)

    # Map 'JPOL' to 'JPOL' and 'OTHER' to 'not_JPOL'
    #res_df['label'] = res_df['label'].map({'Y': 'JPOL', 'N': 'not_JPOL'})
    res_df['label'] = res_df['label'].map({'JPOL': 'JPOL', 'OTHER': 'not_JPOL'})

    # Merge the two DataFrames on 'paragraph_number' and 'file_name'
    merged_df = pd.merge(res_df, gt_df, on=['paragraph_number', 'file_name'], how='outer')

    # Fill missing predictions (in the 'label' column) with 'not_JPOL'
    merged_df['label'] = merged_df['label'].fillna('not_JPOL')

    # Create binary labels for metric calculations
    merged_df['ground_truth_binary'] = merged_df['ground_truth'].apply(lambda x: 1 if x == 'JPOL' else 0)
    merged_df['predicted_binary'] = merged_df['label'].apply(lambda x: 1 if x == 'JPOL' else 0)

    # Function to calculate the metrics for each group (by file_name)
    def calculate_metrics(group):
        precision = precision_score(group['ground_truth_binary'], group['predicted_binary'], zero_division=0)
        recall = recall_score(group['ground_truth_binary'], group['predicted_binary'], zero_division=0)
        f1 = f1_score(group['ground_truth_binary'], group['predicted_binary'], zero_division=0)
        return pd.Series({'precision': precision, 'recall': recall, 'f1': f1})

    # Calculate metrics for each file_name group
    metrics_by_filename = merged_df.groupby('file_name').apply(calculate_metrics).reset_index()

    return metrics_by_filename, merged_df


In [ ]:
import pandas as pd
from sklearn.metrics import precision_score, recall_score, f1_score

def compute_total_metrics(ground_truth_df, results_df):
    res_df = results_df.copy()
    gt_df = ground_truth_df.copy()

    gt_df['paragraph_number'] = gt_df['paragraph_number'].astype(str)
    res_df['paragraph_number'] = res_df['paragraph_number'].astype(str)

    # Map 'JPOL' to 'JPOL' and 'OTHER' to 'not_JPOL'
    res_df['label'] = res_df['label'].map({'JPOL': 'JPOL', 'OTHER': 'not_JPOL'})
    #res_df['label'] = res_df['label'].map({'Y': 'JPOL', 'N': 'not_JPOL'})

    # Merge the two DataFrames on 'paragraph_number' and 'file_name'
    merged_df = pd.merge(res_df, gt_df, on=['paragraph_number', 'file_name'], how='outer')

    # Fill missing predictions (in the 'label' column) with 'not_JPOL'
    merged_df['label'] = merged_df['label'].fillna('not_JPOL')

    # Create binary labels for metric calculations
    merged_df['ground_truth_binary'] = merged_df['ground_truth'].apply(lambda x: 1 if x == 'JPOL' else 0)
    merged_df['predicted_binary'] = merged_df['label'].apply(lambda x: 1 if x == 'JPOL' else 0)

    # Calculate precision, recall, and f1-score
    precision = precision_score(merged_df['ground_truth_binary'], merged_df['predicted_binary'], zero_division=0)
    recall = recall_score(merged_df['ground_truth_binary'], merged_df['predicted_binary'], zero_division=0)
    f1 = f1_score(merged_df['ground_truth_binary'], merged_df['predicted_binary'], zero_division=0)

    return precision, recall, f1

In [ ]:
results_df = pd.read_excel("/content/drive/MyDrive/POLINE/Results/preprocessed_input/Dataset_V1/predictions/billaspries_remaining.xlsx")

# Chiamare la funzione
metrics_by_filename, merged_df = compute_metrics(gt_batch1, results_df)

# Mostrare i risultati
print(metrics_by_filename)

precision, recall, f1 = compute_total_metrics(gt_batch1, results_df)

print(f'<--------------------->')

print(f'Total Precision: {precision:.2f}')
print(f'Total Recall: {recall:.2f}')
print(f'Total F1-Score: {f1:.2f}')

   file_name  precision    recall        f1
0      A & G   1.000000  0.909091  0.952381
1      Autor   0.727273  0.888889  0.800000
2      Boehr   0.636364  1.000000  0.777778
3      CS an   0.818182  0.692308  0.750000
4      DNB B   0.916667  0.687500  0.785714
5      ELVOS   0.444444  1.000000  0.615385
6      Euler   0.777778  1.000000  0.875000
7      Finan   0.000000  0.000000  0.000000
8      I Gmb   0.476190  0.952381  0.634921
9      Micha   0.000000  0.000000  0.000000
10     Minis   0.692308  0.642857  0.666667
<--------------------->
Total Precision: 0.65
Total Recall: 0.66
Total F1-Score: 0.66


<ipython-input-190-17a79ebb1789>:32: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  metrics_by_filename = merged_df.groupby('file_name').apply(calculate_metrics).reset_index()


In [ ]:
metrics_by_filename['total precision'] = precision
metrics_by_filename['total recall'] = recall
metrics_by_filename['total f1'] = f1

In [ ]:
metrics_by_filename.to_excel("/content/drive/MyDrive/POLINE/Results/preprocessed_input/metrics_evaluation/Piera_e_Alessia 24.06.24_1_predictions.xlsx")

# EVALUATION UNSTRUCTURED

TODO BILLI:

*   Destruttura sentenze

TODO ASPRO:

*   Metriche AreaJPOL/AreaTesto
*   Map "5 words tags to paragraph" per comparare approccio strutturato e non






JPOL Ratio= Total number of words in the text / Total number of words tagged as JPOL​

\text{JPOL Ratio} = \frac{W_{\text{JPOL}}}{W_{\text{total}}}


Extract JPOL Sections: Use regex to find the text within JPOL tags.
Count JPOL Words: Split the JPOL text into parts around the dots. Count the words in the first and last parts.
Count Total Words: Remove JPOL tags from the total text and count the remaining words.
Add JPOL Word Count: Add the JPOL word count to the total word count.
Calculate Ratio: Compute the ratio of JPOL words to the total words.

In [ ]:
import re

def calculate_jpol_ratio(tagged_phrase, total_text, jpol_tag='<JPOL>', jpol_end_tag='</JPOL>'):
    # Extract JPOL sections
    jpol_texts = re.findall(f'{jpol_tag}(.*?){jpol_end_tag}', tagged_phrase, re.DOTALL)

    # Split JPOL text into beginning and end parts
    jpol_word_count = 0
    for jpol in jpol_texts:
        # Split into parts around the dots
        parts = jpol.split('...')
        if len(parts) == 2:
            # Count words in the first and last parts
            first_part_words = len(parts[0].split())
            last_part_words = len(parts[1].split())
            jpol_word_count += first_part_words + last_part_words

    # Remove JPOL tags from text to count total words
    clean_text = re.sub(f'{jpol_tag}.*?{jpol_end_tag}', '', total_text, flags=re.DOTALL)
    total_word_count = len(clean_text.split())

    # Add the JPOL word count to the total word count
    total_word_count += jpol_word_count

    # Calculate JPOL Ratio
    if total_word_count == 0:
        return 0

    jpol_ratio = jpol_word_count / total_word_count
    return round(jpol_ratio, 2)

In [ ]:
from docx import Document

f = open("/content/drive/MyDrive/POLINE/Results/preprocessed_input/full_responses_billaspries_4_no_par/A & G Fahrschul-Akademie GmbH v Finanzamt Wolfenbüttel.xml_full_response.docx", 'rb')
document = Document(f)
f.close()

jpols_text = document.paragraphs[0].text

file = "/content/drive/MyDrive/POLINE/Dataset/Dataset_V1/Judgements_Subset/A & G Fahrschul-Akademie GmbH v Finanzamt Wolfenbüttel.xml"

with open(file) as f:
    file = f.read()

judg_text = "".join(file)

judg_text = tp.extract_text_between_markers(judg_text)

In [ ]:
judg_text

'The first question\n\n16\n\nBy its first question, the referring court asks, in essence, whether the concept of ‘school or university education’, within the meaning of Article 132(1)(i) and (j) of Directive 2006/112, must be interpreted as covering motor vehicle driving tuition provided by a driving school, such as that at issue in the main proceedings, for the purpose of acquiring driving licences for vehicles in categories B and C1 referred to in Article 4(4) of Directive 2006/126.\n\n17\n\nArticle 132 of Directive 2006/112 provides for exemptions which, as indicated by the title of the chapter in which that provision features, are intended to encourage certain activities in the public interest. However, those exemptions do not cover every activity performed in the public interest, but only those listed in that provision and described in great detail (judgment of 4 May 2017, Brockenhurst College, C-699/15, EU:C:2017:344, paragraph 22 and the case-law cited).\n\n18\n\nAccording to th

In [ ]:
ratio = calculate_jpol_ratio(jpols_text, judg_text)
print(ratio)

0.13
